In [ ]:
# ============================================================
# CELL 1 — SETUP
# ============================================================
!pip install statsmodels linearmodels -q

import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/ECB_Research/ECB_Project'
DATA_PATH  = f'{DRIVE_ROOT}/output/lp_dataset.csv'

print("Setup completo")

In [ ]:
# ============================================================
# CELL 2 — CARICA DATI
# ============================================================
df = pd.read_csv(DATA_PATH, parse_dates=['meeting_date'])
df = df.sort_values('meeting_date').reset_index(drop=True)

df['surprise']   = df['dfr_surprise_mech']
df['shock_pos']  = df['surprise'].clip(lower=0)
df['shock_neg']  = df['surprise'].clip(upper=0)
df['t']          = np.arange(len(df))

print(f"Dataset: {len(df)} meeting ({df['meeting_date'].min().date()} -> {df['meeting_date'].max().date()})")
print(f"\nVariabili disponibili:")
print(f"  surprise:   mean={df['surprise'].mean():.2f}bp, std={df['surprise'].std():.2f}bp")
print(f"  hicp_yoy:   mean={df['hicp_yoy'].mean():.2f}%, std={df['hicp_yoy'].std():.2f}%")
print(f"  gdp_yoy:    mean={df['gdp_yoy'].mean():.2f}%, std={df['gdp_yoy'].std():.2f}%")
print(f"  unemp_rate: mean={df['unemp_rate'].mean():.2f}%, std={df['unemp_rate'].std():.2f}%")

In [ ]:
# ============================================================
# CELL 3 — LOCAL PROJECTIONS (Jorda 2005)
# ============================================================

def local_projection(df, y_var, shock_var, controls, H=12, lags=2):
    results = []
    for h in range(H + 1):
        rows = []
        for i in range(lags, len(df) - h):
            y_future = df[y_var].iloc[i + h]
            y_past   = df[y_var].iloc[i - 1]
            if pd.isna(y_future) or pd.isna(y_past):
                continue
            dep_var = y_future - y_past
            shock   = df[shock_var].iloc[i]
            row = {'dep': dep_var, 'shock': shock}
            for c in controls:
                row[c] = df[c].iloc[i]
            for lag in range(1, lags + 1):
                y_lag = df[y_var].iloc[i - lag]
                y_lag_prev = df[y_var].iloc[i - lag - 1] if i - lag - 1 >= 0 else np.nan
                row[f'dy_lag{lag}'] = y_lag - y_lag_prev if not pd.isna(y_lag) and not pd.isna(y_lag_prev) else 0
            rows.append(row)
        if not rows:
            continue
        reg_df = pd.DataFrame(rows).dropna()
        if len(reg_df) < 20:
            continue
        y = reg_df['dep']
        X_cols = ['shock'] + controls + [f'dy_lag{l}' for l in range(1, lags + 1)]
        X_cols = [c for c in X_cols if c in reg_df.columns]
        X = sm.add_constant(reg_df[X_cols])
        model  = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': h + 1})
        beta = model.params['shock']
        se   = model.bse['shock']
        results.append({
            'h': h, 'beta': beta, 'se': se,
            'ci_lower': beta - 1.96 * se, 'ci_upper': beta + 1.96 * se,
            'ci_lower_90': beta - 1.645 * se, 'ci_upper_90': beta + 1.645 * se,
            'nobs': int(model.nobs), 'pvalue': model.pvalues['shock']
        })
    return pd.DataFrame(results)

controls = ['hicp_yoy', 'gdp_yoy']

print("Stimando LP per HICP...")
lp_hicp = local_projection(df=df, y_var='hicp_yoy', shock_var='surprise', controls=controls, H=12, lags=2)

print("Stimando LP per GDP...")
lp_gdp = local_projection(df=df, y_var='gdp_yoy', shock_var='surprise', controls=['hicp_yoy', 'gdp_yoy'], H=12, lags=2)

print(f"\nLP HICP stimata su {lp_hicp['nobs'].mean():.0f} obs in media")
print(f"LP GDP  stimata su {lp_gdp['nobs'].mean():.0f} obs in media")
print(f"\nCoefficienti HICP (beta_h):")
print(lp_hicp[['h','beta','se','pvalue']].round(4).to_string(index=False))

In [ ]:
# ============================================================
# CELL 5 — LP A FREQUENZA MENSILE (metodologia corretta)
# ============================================================

monthly_index = pd.date_range('1999-01-31', '2025-12-31', freq='ME')
monthly = pd.DataFrame({'date': monthly_index})

hicp_m = df[['meeting_date', 'hicp_yoy']].copy()
hicp_m['date'] = hicp_m['meeting_date'] + pd.offsets.MonthEnd(0)
hicp_m = hicp_m.groupby('date')['hicp_yoy'].first().reset_index()

gdp_m = df[['meeting_date', 'gdp_yoy']].copy()
gdp_m['date'] = gdp_m['meeting_date'] + pd.offsets.MonthEnd(0)
gdp_m = gdp_m.groupby('date')['gdp_yoy'].first().reset_index()

unemp_m = df[['meeting_date', 'unemp_rate']].copy()
unemp_m['date'] = unemp_m['meeting_date'] + pd.offsets.MonthEnd(0)
unemp_m = unemp_m.groupby('date')['unemp_rate'].first().reset_index()

surprise_m = df[['meeting_date', 'dfr_surprise_mech']].copy()
surprise_m['date'] = surprise_m['meeting_date'] + pd.offsets.MonthEnd(0)
surprise_m = surprise_m.groupby('date')['dfr_surprise_mech'].sum().reset_index()
surprise_m.columns = ['date', 'surprise']

monthly = monthly.merge(surprise_m, on='date', how='left')
monthly['surprise'] = monthly['surprise'].fillna(0)
monthly = monthly.merge(hicp_m, on='date', how='left')
monthly = monthly.merge(gdp_m, on='date', how='left')
monthly = monthly.merge(unemp_m, on='date', how='left')

monthly['gdp_yoy'] = monthly['gdp_yoy'].ffill()

monthly['shock_pos'] = monthly['surprise'].clip(lower=0)
monthly['shock_neg'] = monthly['surprise'].clip(upper=0)

print(f"Dataset mensile: {len(monthly)} mesi")
print(f"   Mesi con almeno un meeting: {(monthly['surprise'] != 0).sum()}")
print(f"   Mesi senza meeting: {(monthly['surprise'] == 0).sum()}")
print(f"\n   Null per variabile:")
for col in ['hicp_yoy', 'gdp_yoy', 'unemp_rate', 'surprise']:
    print(f"     {col}: {monthly[col].isna().sum()}")

In [ ]:
# ============================================================
# CELL 6 — STIMA LP MENSILE
# ============================================================

def lp_monthly(data, y_var, shock_var, controls, H=24, lags=3):
    results = []
    for h in range(H + 1):
        rows = []
        for i in range(lags, len(data) - h):
            y_future = data[y_var].iloc[i + h]
            y_past   = data[y_var].iloc[i - 1]
            if pd.isna(y_future) or pd.isna(y_past):
                continue
            row = {'dep': y_future - y_past, shock_var: data[shock_var].iloc[i]}
            for c in controls:
                val = data[c].iloc[i]
                if pd.isna(val):
                    row = None
                    break
                row[c] = val
            if row is None:
                continue
            for lag in range(1, lags + 1):
                y_lag      = data[y_var].iloc[i - lag]
                y_lag_prev = data[y_var].iloc[i - lag - 1] if i - lag - 1 >= 0 else np.nan
                row[f'dy_lag{lag}'] = y_lag - y_lag_prev if not pd.isna(y_lag) and not pd.isna(y_lag_prev) else 0
            rows.append(row)
        reg_df = pd.DataFrame(rows).dropna()
        if len(reg_df) < 30:
            continue
        y = reg_df['dep']
        X_cols = [shock_var] + controls + [f'dy_lag{l}' for l in range(1, lags + 1)]
        X_cols = [c for c in X_cols if c in reg_df.columns]
        X = sm.add_constant(reg_df[X_cols])
        model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': max(h, 1)})
        beta = model.params[shock_var]
        se   = model.bse[shock_var]
        results.append({
            'h': h, 'beta': beta, 'se': se,
            'ci_lower': beta - 1.96 * se, 'ci_upper': beta + 1.96 * se,
            'ci_lower_90': beta - 1.645 * se, 'ci_upper_90': beta + 1.645 * se,
            'nobs': int(model.nobs), 'pvalue': model.pvalues[shock_var]
        })
    return pd.DataFrame(results)

controls = ['hicp_yoy', 'gdp_yoy']

print("Stimando LP mensile - HICP (simmetrica)...")
lp_hicp_m = lp_monthly(monthly, 'hicp_yoy', 'surprise', controls, H=24, lags=3)

print("Stimando LP mensile - GDP (simmetrica)...")
lp_gdp_m = lp_monthly(monthly, 'gdp_yoy', 'surprise', controls, H=24, lags=3)

print("Stimando LP mensile - HICP (hawkish)...")
lp_hicp_pos = lp_monthly(monthly, 'hicp_yoy', 'shock_pos', controls, H=24, lags=3)

print("Stimando LP mensile - HICP (dovish)...")
lp_hicp_neg = lp_monthly(monthly, 'hicp_yoy', 'shock_neg', controls, H=24, lags=3)

print("Stimando LP mensile - GDP (hawkish)...")
lp_gdp_pos = lp_monthly(monthly, 'gdp_yoy', 'shock_pos', controls, H=24, lags=3)

print("Stimando LP mensile - GDP (dovish)...")
lp_gdp_neg = lp_monthly(monthly, 'gdp_yoy', 'shock_neg', controls, H=24, lags=3)

print(f"\nStime completate")
print(f"   HICP: {len(lp_hicp_m)} orizzonti, media obs={lp_hicp_m['nobs'].mean():.0f}")
print(f"   GDP:  {len(lp_gdp_m)} orizzonti, media obs={lp_gdp_m['nobs'].mean():.0f}")
print(f"\nCoefficienti HICP - primi 12 mesi:")
print(lp_hicp_m[['h','beta','se','pvalue']].head(13).round(5).to_string(index=False))

In [ ]:
# ============================================================
# CELL 7 — NORMALIZZAZIONE + GRAFICI IRF
# ============================================================

scale = 25.0
monthly['surprise_n']  = monthly['surprise']  / scale
monthly['shock_pos_n'] = monthly['shock_pos'] / scale
monthly['shock_neg_n'] = monthly['shock_neg'] / scale

controls = ['hicp_yoy', 'gdp_yoy']

print("Stimando LP normalizzate...")
lp_hicp_n     = lp_monthly(monthly, 'hicp_yoy', 'surprise_n',  controls, H=24, lags=3)
lp_gdp_n      = lp_monthly(monthly, 'gdp_yoy',  'surprise_n',  controls, H=24, lags=3)
lp_hicp_pos_n = lp_monthly(monthly, 'hicp_yoy', 'shock_pos_n', controls, H=24, lags=3)
lp_hicp_neg_n = lp_monthly(monthly, 'hicp_yoy', 'shock_neg_n', controls, H=24, lags=3)
lp_gdp_pos_n  = lp_monthly(monthly, 'gdp_yoy',  'shock_pos_n', controls, H=24, lags=3)
lp_gdp_neg_n  = lp_monthly(monthly, 'gdp_yoy',  'shock_neg_n', controls, H=24, lags=3)

print("Stime completate")

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('ECB Monetary Policy Surprises - Impulse Response Functions\n'
             'Local Projections (Jorda 2005) | Shock = +25bp hawkish surprise | HAC SE',
             fontsize=13, fontweight='bold')

def plot_irf(ax, lp, title, ylabel, color='steelblue'):
    ax.fill_between(lp['h'], lp['ci_lower'], lp['ci_upper'], alpha=0.15, color=color)
    ax.fill_between(lp['h'], lp['ci_lower_90'], lp['ci_upper_90'], alpha=0.25, color=color)
    ax.plot(lp['h'], lp['beta'], color=color, linewidth=2, marker='o', markersize=4, label='Point estimate')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.6)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Months after shock', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(range(0, 25, 3))

def plot_irf_asym(ax, lp_pos, lp_neg, title, ylabel):
    ax.fill_between(lp_pos['h'], lp_pos['ci_lower'], lp_pos['ci_upper'], alpha=0.12, color='red')
    ax.plot(lp_pos['h'], lp_pos['beta'], color='red', linewidth=2, marker='o', markersize=4, label='Hawkish (+25bp)')
    ax.fill_between(lp_neg['h'], lp_neg['ci_lower'], lp_neg['ci_upper'], alpha=0.12, color='blue')
    ax.plot(lp_neg['h'], lp_neg['beta'], color='blue', linewidth=2, marker='s', markersize=4, label='Dovish (-25bp)')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.6)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Months after shock', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(range(0, 25, 3))

plot_irf(axes[0,0], lp_hicp_n, 'IRF: HICP Inflation (symmetric)', 'Delta HICP (pp)', color='steelblue')
plot_irf(axes[0,1], lp_gdp_n, 'IRF: GDP Growth (symmetric)', 'Delta GDP yoy (pp)', color='darkgreen')
plot_irf_asym(axes[1,0], lp_hicp_pos_n, lp_hicp_neg_n, 'IRF: HICP Inflation (asymmetric)', 'Delta HICP (pp)')
plot_irf_asym(axes[1,1], lp_gdp_pos_n, lp_gdp_neg_n, 'IRF: GDP Growth (asymmetric)', 'Delta GDP yoy (pp)')

plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/output/lp_irfs_monthly.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafici salvati su Drive")

In [ ]:
# ============================================================
# CELL 8 — WALD TEST ASIMMETRIA + CONTROLLO ENDOGENEITA
# ============================================================

from scipy.stats import chi2

def lp_monthly_joint(data, y_var, controls, H=24, lags=3):
    results = []
    for h in range(H + 1):
        rows = []
        for i in range(lags, len(data) - h):
            y_future = data[y_var].iloc[i + h]
            y_past   = data[y_var].iloc[i - 1]
            if pd.isna(y_future) or pd.isna(y_past):
                continue
            row = {
                'dep': y_future - y_past,
                'shock_pos': data['shock_pos_n'].iloc[i],
                'shock_neg': data['shock_neg_n'].iloc[i],
            }
            skip = False
            for c in controls:
                val = data[c].iloc[i]
                if pd.isna(val):
                    skip = True
                    break
                row[c] = val
            if skip:
                continue
            for lag in range(1, lags + 1):
                y_lag      = data[y_var].iloc[i - lag]
                y_lag_prev = data[y_var].iloc[i - lag - 1] if i - lag - 1 >= 0 else np.nan
                row[f'dy_lag{lag}'] = y_lag - y_lag_prev if not pd.isna(y_lag) and not pd.isna(y_lag_prev) else 0
            rows.append(row)
        reg_df = pd.DataFrame(rows).dropna()
        if len(reg_df) < 30:
            continue
        y = reg_df['dep']
        X_cols = ['shock_pos', 'shock_neg'] + controls + [f'dy_lag{l}' for l in range(1, lags + 1)]
        X_cols = [c for c in X_cols if c in reg_df.columns]
        X = sm.add_constant(reg_df[X_cols])
        model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': max(h, 1)})
        b_pos  = model.params['shock_pos']
        b_neg  = model.params['shock_neg']
        se_pos = model.bse['shock_pos']
        se_neg = model.bse['shock_neg']
        cov_matrix = model.cov_params()
        R = np.array([[1, -1]])
        idx_pos = list(model.params.index).index('shock_pos')
        idx_neg = list(model.params.index).index('shock_neg')
        V = cov_matrix.iloc[[idx_pos, idx_neg], [idx_pos, idx_neg]].values
        b_diff = b_pos - b_neg
        wald_stat = float((b_diff**2) / (R @ V @ R.T).item())
        wald_pval = 1 - chi2.cdf(wald_stat, df=1)
        results.append({
            'h': h, 'beta_pos': b_pos, 'beta_neg': b_neg, 'se_pos': se_pos, 'se_neg': se_neg,
            'ci_pos_lo': b_pos - 1.96 * se_pos, 'ci_pos_hi': b_pos + 1.96 * se_pos,
            'ci_neg_lo': b_neg - 1.96 * se_neg, 'ci_neg_hi': b_neg + 1.96 * se_neg,
            'wald_stat': wald_stat, 'wald_pval': wald_pval,
            'asymmetric': wald_pval < 0.10, 'nobs': int(model.nobs),
        })
    return pd.DataFrame(results)

monthly['gdp_trend']    = monthly['gdp_yoy'].rolling(12, min_periods=6).mean()
monthly['gdp_gap']      = monthly['gdp_yoy'] - monthly['gdp_trend']

controls_ext = ['hicp_yoy', 'gdp_yoy', 'gdp_gap']

print("Stimando LP congiunta HICP con controllo endogeneita...")
lp_hicp_joint = lp_monthly_joint(monthly, 'hicp_yoy', controls_ext, H=24, lags=3)

print("Stimando LP congiunta GDP con controllo endogeneita...")
lp_gdp_joint  = lp_monthly_joint(monthly, 'gdp_yoy',  controls_ext, H=24, lags=3)

print("\nStime completate")
print(f"\n{'='*65}")
print("WALD TEST - H0: beta_hawkish = beta_dovish (asimmetria = 0)")
print(f"{'='*65}")
print(f"\nHICP - orizzonti con asimmetria significativa (p<0.10):")
sig = lp_hicp_joint[lp_hicp_joint['asymmetric']]
if len(sig) > 0:
    print(sig[['h','beta_pos','beta_neg','wald_stat','wald_pval']].round(4).to_string(index=False))
else:
    print("  Nessun orizzonte significativo al 10%")

print(f"\nGDP - orizzonti con asimmetria significativa (p<0.10):")
sig_g = lp_gdp_joint[lp_gdp_joint['asymmetric']]
if len(sig_g) > 0:
    print(sig_g[['h','beta_pos','beta_neg','wald_stat','wald_pval']].round(4).to_string(index=False))
else:
    print("  Nessun orizzonte significativo al 10%")

In [ ]:
# ============================================================
# CELL 9 — GRAFICI FINALI CON WALD TEST
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('ECB Monetary Policy Surprises - Asymmetric Transmission\n'
             'Local Projections (Jorda 2005) | Shock = +/-25bp | HAC SE | Wald test asymmetry',
             fontsize=13, fontweight='bold')

def plot_asym_wald(ax, lp, title, ylabel):
    h       = lp['h']
    b_pos   = lp['beta_pos']
    b_neg   = lp['beta_neg']
    sig     = lp['asymmetric']
    ax.fill_between(h, lp['ci_pos_lo'], lp['ci_pos_hi'], alpha=0.12, color='red')
    ax.fill_between(h, lp['ci_neg_lo'], lp['ci_neg_hi'], alpha=0.12, color='blue')
    ax.plot(h, b_pos, color='red',  linewidth=2, marker='o', markersize=4, label='Hawkish surprise (+25bp)')
    ax.plot(h, b_neg, color='blue', linewidth=2, marker='s', markersize=4, label='Dovish surprise (-25bp)')
    for _, row in lp[sig].iterrows():
        ax.axvspan(row['h'] - 0.4, row['h'] + 0.4, alpha=0.08, color='gold', zorder=0)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.6)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Months after shock', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(range(0, 25, 3))
    n_sig = sig.sum()
    ax.text(0.02, 0.02, f'Yellow bands: Wald test asymmetry p<0.10\n({n_sig} horizons significant)',
            transform=ax.transAxes, fontsize=9, bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plot_asym_wald(axes[0], lp_hicp_joint, 'Asymmetric IRF: HICP Inflation', 'Delta HICP (pp per +25bp shock)')
plot_asym_wald(axes[1], lp_gdp_joint, 'Asymmetric IRF: GDP Growth', 'Delta GDP yoy (pp per +25bp shock)')

plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/output/lp_asymmetric_wald.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafici finali salvati su Drive")

print("\n" + "="*60)
print("RISULTATI CHIAVE - TIER 1")
print("="*60)
print(f"\nEffetto di uno shock hawkish da +25bp sull'inflazione HICP:")
for h in [6, 9, 12, 18, 24]:
    row = lp_hicp_joint[lp_hicp_joint['h'] == h]
    if not row.empty:
        r = row.iloc[0]
        sig_str = "***" if r['wald_pval'] < 0.01 else "**" if r['wald_pval'] < 0.05 else "*" if r['wald_pval'] < 0.10 else ""
        print(f"  h={h:2d} mesi: hawkish={r['beta_pos']:.3f}pp  dovish={r['beta_neg']:.3f}pp  Wald p={r['wald_pval']:.3f} {sig_str}")

In [ ]:
# ============================================================
# CELL 10 — STRUTTURA DEL BREAK 2015: MEDIA, VARIANZA, DISTRIBUZIONE, CHOW
# ============================================================
import statsmodels.api as sm
from scipy import stats

surprises = pd.read_csv(f'{DRIVE_ROOT}/output/surprises_timeseries.csv', parse_dates=['meeting_date'])
surprises = surprises.sort_values('meeting_date').reset_index(drop=True)

surprises['post2015'] = (surprises['meeting_date'] >= '2015-01-01').astype(int)
surprises['t'] = np.arange(len(surprises))

pre  = surprises[surprises['post2015'] == 0]['dfr_surprise_mech']
post = surprises[surprises['post2015'] == 1]['dfr_surprise_mech']

# --- Test 1: differenza nella media (t-test / F-test) ---
tstat, pval = stats.ttest_ind(pre, post)
print("=== TEST DIFFERENZA MEDIE ===")
print(f"Pre-2015  (n={len(pre)}):  mean={pre.mean():.3f}bp, std={pre.std():.3f}bp")
print(f"Post-2015 (n={len(post)}): mean={post.mean():.3f}bp, std={post.std():.3f}bp")
print(f"t-stat={tstat:.3f}, p-value={pval:.4f}")

# --- Test 2: differenza nella varianza (Levene) ---
lstat, lpval = stats.levene(pre, post)
print(f"\n=== TEST DIFFERENZA VARIANZE (Levene) ===")
print(f"Pre-2015  std={pre.std():.3f}bp")
print(f"Post-2015 std={post.std():.3f}bp")
print(f"Levene stat={lstat:.3f}, p-value={lpval:.4f}")

# --- Test 3: Kolmogorov-Smirnov e Mann-Whitney (test di distribuzione) ---
# Aggiunto su richiesta del relatore: il t-test/F-test sulla sola media non e'
# sufficiente quando la media cambia anche di segno (pre: -3.7bp, post: +5.0bp).
# KS e Mann-Whitney testano l'uguaglianza dell'intera distribuzione, senza
# richiedere l'assunzione di normalita' sottostante al t-test/F-test.
ks_stat, ks_pval = stats.ks_2samp(pre, post)
print(f"\n=== KOLMOGOROV-SMIRNOV (test di distribuzione) ===")
print(f"KS statistic: {ks_stat:.4f}")
print(f"p-value:      {ks_pval:.6f}")

mw_stat, mw_pval = stats.mannwhitneyu(pre, post, alternative='two-sided')
print(f"\n=== MANN-WHITNEY U (test di distribuzione, rank-based) ===")
print(f"U statistic: {mw_stat:.1f}")
print(f"p-value:     {mw_pval:.6f}")

print(f"\nMediane: pre-2015={pre.median():.2f}bp, post-2015={post.median():.2f}bp")

# --- Test 4: Chow test formale (break nel trend) ---
y = surprises['dfr_surprise_mech'].values
t = surprises['t'].values
D = surprises['post2015'].values

X_unres = sm.add_constant(np.column_stack([t, D, D*t]))
model_unres = sm.OLS(y, X_unres).fit()
rss_unres = model_unres.ssr

X_res = sm.add_constant(t)
model_res = sm.OLS(y, X_res).fit()
rss_res = model_res.ssr

k = 2
n = len(y)
chow_stat = ((rss_res - rss_unres) / k) / (rss_unres / (n - 2*k))
chow_pval = 1 - stats.f.cdf(chow_stat, k, n - 2*k)

print(f"\n=== CHOW TEST (break a gennaio 2015, break nel trend) ===")
print(f"F-stat={chow_stat:.3f}, p-value={chow_pval:.4f}")
if chow_pval < 0.05:
    print("-> Break strutturale SIGNIFICATIVO al 5%")
elif chow_pval < 0.10:
    print("-> Break strutturale significativo al 10%")
else:
    print("-> Nessun break strutturale significativo")

print(f"\n{'='*60}")
print("RIEPILOGO PER LA TESI (Sezione 3.6)")
print(f"{'='*60}")
print("F-test (mean shift), KS, Mann-Whitney tutti rifiutano l'uguaglianza")
print("tra le due sotto-distribuzioni; Levene non rifiuta l'omoschedasticita'.")
print("Interpretazione: il break e' reale e riguarda l'intera distribuzione,")
print("non solo la media, ma la dispersione (varianza) resta stabile.")

In [ ]:
# ============================================================
# CELL 11 — LP CON DUMMY POST-2015 (robustness)
# ============================================================

monthly['post2015'] = (monthly['date'] >= '2015-01-01').astype(int)
controls_robust = ['hicp_yoy', 'gdp_yoy', 'gdp_gap', 'post2015']

print("Stimando LP robusta con dummy post-2015...")
lp_hicp_robust = lp_monthly_joint(monthly, 'hicp_yoy', controls_robust, H=24, lags=3)

print("\nCompletata")
print(f"\nConfronto risultati chiave (h=9):")
base = lp_hicp_joint[lp_hicp_joint['h']==9].iloc[0]
rob  = lp_hicp_robust[lp_hicp_robust['h']==9].iloc[0]
print(f"  Baseline:     hawkish={base['beta_pos']:.3f}pp, dovish={base['beta_neg']:.3f}pp, Wald p={base['wald_pval']:.4f}")
print(f"  Con dummy:    hawkish={rob['beta_pos']:.3f}pp,  dovish={rob['beta_neg']:.3f}pp,  Wald p={rob['wald_pval']:.4f}")
print(f"\n-> Se i risultati sono simili, il break non distorce le stime LP")

In [ ]:
# ============================================================
# CELL 12 — LP SENZA PIL CONTEMPORANEO (solo lag) — bad-control check
# ============================================================

monthly['gdp_lag1'] = monthly['gdp_yoy'].shift(1)
monthly['gdp_lag2'] = monthly['gdp_yoy'].shift(2)
monthly['gdp_lag3'] = monthly['gdp_yoy'].shift(3)

controls_nolah = ['hicp_yoy', 'gdp_lag1', 'gdp_lag2', 'post2015']

print("Stimando LP senza PIL contemporaneo...")
lp_hicp_nolah = lp_monthly_joint(monthly, 'hicp_yoy', controls_nolah, H=24, lags=3)

print("\nCompletata")
print(f"\nConfronto a h=9:")
base  = lp_hicp_joint[lp_hicp_joint['h']==9].iloc[0]
nolah = lp_hicp_nolah[lp_hicp_nolah['h']==9].iloc[0]
print(f"  Baseline (GDP contemp.): hawkish={base['beta_pos']:.3f}pp, dovish={base['beta_neg']:.3f}pp, Wald p={base['wald_pval']:.4f}")
print(f"  Solo lag GDP:            hawkish={nolah['beta_pos']:.3f}pp, dovish={nolah['beta_neg']:.3f}pp, Wald p={nolah['wald_pval']:.4f}")

In [ ]:
# ============================================================
# CELL 13 — TEST INTERAZIONE SHOCK x DUMMY POST-2015 (regime interaction)
# ============================================================

def lp_monthly_interaction(data, y_var, controls, H=24, lags=3):
    results = []
    for h in range(H + 1):
        rows = []
        for i in range(lags, len(data) - h):
            y_future = data[y_var].iloc[i + h]
            y_past   = data[y_var].iloc[i - 1]
            if pd.isna(y_future) or pd.isna(y_past):
                continue
            s_pos = data['shock_pos_n'].iloc[i]
            s_neg = data['shock_neg_n'].iloc[i]
            d     = data['post2015'].iloc[i]
            row = {
                'dep': y_future - y_past, 'shock_pos': s_pos, 'shock_neg': s_neg,
                'post2015': d, 'int_pos': s_pos * d, 'int_neg': s_neg * d,
            }
            skip = False
            for c in controls:
                val = data[c].iloc[i]
                if pd.isna(val):
                    skip = True
                    break
                row[c] = val
            if skip:
                continue
            for lag in range(1, lags + 1):
                y_lag = data[y_var].iloc[i - lag]
                y_lag_prev = data[y_var].iloc[i - lag - 1] if i - lag - 1 >= 0 else np.nan
                row[f'dy_lag{lag}'] = y_lag - y_lag_prev if not pd.isna(y_lag) and not pd.isna(y_lag_prev) else 0
            rows.append(row)
        reg_df = pd.DataFrame(rows).dropna()
        if len(reg_df) < 30:
            continue
        y = reg_df['dep']
        X_cols = ['shock_pos', 'shock_neg', 'post2015', 'int_pos', 'int_neg'] + controls + [f'dy_lag{l}' for l in range(1, lags + 1)]
        X_cols = [c for c in X_cols if c in reg_df.columns]
        X = sm.add_constant(reg_df[X_cols])
        model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': max(h, 1)})
        results.append({
            'h': h, 'beta_pos': model.params['shock_pos'], 'beta_neg': model.params['shock_neg'],
            'beta_int_pos': model.params['int_pos'], 'beta_int_neg': model.params['int_neg'],
            'p_int_pos': model.pvalues['int_pos'], 'p_int_neg': model.pvalues['int_neg'],
            'nobs': int(model.nobs),
        })
    return pd.DataFrame(results)

controls_lag = ['hicp_yoy', 'gdp_lag1', 'gdp_lag2']

print("Stimando LP con interazione shock x post-2015...")
lp_interaction = lp_monthly_interaction(monthly, 'hicp_yoy', controls_lag, H=24, lags=3)

print("\nCompletata")
print(f"\n{'h':>3} {'beta_pos':>10} {'int_pos':>10} {'p_int':>10} {'sig':>5}")
print("-" * 45)
for _, row in lp_interaction.iterrows():
    sig = "***" if row['p_int_pos'] < 0.01 else "**" if row['p_int_pos'] < 0.05 else "*" if row['p_int_pos'] < 0.10 else ""
    print(f"{int(row['h']):>3} {row['beta_pos']:>10.3f} {row['beta_int_pos']:>10.3f} {row['p_int_pos']:>10.4f} {sig:>5}")

In [ ]:
# ============================================================
# CELL 14 — ROBUSTNESS: ESCLUSIONE 2022-2023 (aggiunta)
# ============================================================
# Verifica che l'asimmetria non sia trainata dalle due grandi sorprese
# hawkish di settembre/ottobre 2022 (+75bp). Escludiamo interamente
# il 2022 e il 2023 e ristimiamo la LP congiunta con Wald test.

monthly_excl = monthly[(monthly['date'] < '2022-01-01') | (monthly['date'] >= '2024-01-01')].reset_index(drop=True)

print(f"Campione con 2022-2023 escluso: {len(monthly_excl)} mesi (da {len(monthly)})")

print("\nStimando LP congiunta HICP escludendo 2022-2023...")
lp_hicp_excl2022 = lp_monthly_joint(monthly_excl, 'hicp_yoy', controls_ext, H=24, lags=3)

print("\nCompletata")
print(f"\nConfronto a h=9 (orizzonte di picco):")
base = lp_hicp_joint[lp_hicp_joint['h']==9].iloc[0]
excl = lp_hicp_excl2022[lp_hicp_excl2022['h']==9].iloc[0]
print(f"  Full sample:          hawkish={base['beta_pos']:.3f}pp, dovish={base['beta_neg']:.3f}pp, "
      f"ratio={abs(base['beta_pos'])/base['beta_neg']:.2f}:1, Wald p={base['wald_pval']:.4f}")
print(f"  Escl. 2022-2023:      hawkish={excl['beta_pos']:.3f}pp, dovish={excl['beta_neg']:.3f}pp, "
      f"ratio={abs(excl['beta_pos'])/excl['beta_neg']:.2f}:1, Wald p={excl['wald_pval']:.4f}")
print(f"\nNota interpretativa: il coefficiente hawkish si RAFFORZA (non si indebolisce)")
print(f"escludendo il 2022-2023 -> l'asimmetria non e' un artefatto di quel biennio.")
print(f"Il coefficiente dovish, pero', si dimezza pur non contenendo quasi sorprese")
print(f"dovish nel biennio escluso: i due coefficienti non sono identificati in modo")
print(f"indipendente. Il rapporto 1.8:1 (full sample) resta la stima preferita; il")
print(f"rapporto piu' ampio ottenuto qui va letto come sensitivity check, non come")
print(f"conferma indipendente di un effetto strutturale piu' grande.")

In [ ]:
# ============================================================
# CELL 15 — MODAL ACCURACY vs BENCHMARK NAIVE (aggiunta)
# ============================================================
# Confronta l'accuratezza del bin modale del pipeline con due benchmark
# naive: 'always predict hold' e 'random walk' (ripeti l'ultima decisione).
# Aggiunto su richiesta del relatore (perche' 'always hold' e' naive
# rispetto a 'same decision as the last one'?).

df_sorted = df.sort_values('meeting_date').reset_index(drop=True).copy()

# --- Always-hold benchmark ---
hold_accuracy = (df_sorted['dfr_actual_bp'] == 0).mean()

# --- Random-walk benchmark: predici la stessa decisione della riunione precedente ---
df_sorted['prev_decision'] = df_sorted['dfr_actual_bp'].shift(1)
rw_valid = df_sorted['prev_decision'].notna()
rw_accuracy = (df_sorted.loc[rw_valid, 'dfr_actual_bp'] == df_sorted.loc[rw_valid, 'prev_decision']).mean()

print("=== BENCHMARK NAIVE vs PIPELINE ===")
print(f"Always predict hold:        {hold_accuracy:.1%}")
print(f"Random walk (repeat last):  {rw_accuracy:.1%}")
print(f"Pipeline (modal accuracy):  vedi modal_outcome vs direction in surprises_timeseries.csv")
print()
print("Nota interpretativa: entrambi i benchmark naive battono la modal accuracy")
print("del pipeline (71%) su questa specifica metrica. Non e' un difetto del")
print("modello: gli hold dominano il campione (81.4%), quindi 'always hold' e'")
print("quasi imbattibile sulla point accuracy per costruzione. Il pipeline non e'")
print("ottimizzato per questa metrica, ma per produrre un'aspettativa CONTINUA")
print("ben calibrata (E[Delta_i]), usata per calcolare la sorpresa. Una regola")
print("'always hold' genererebbe una sorpresa implausibile ad ogni mossa telegrafata")
print("(es. giugno 2024: sorpresa reale -0.4bp, la regola naive direbbe -25bp).")

In [ ]:
# ============================================================
# CELL 16 — FIGURA SINGOLO PANNELLO PER LA PRESENTAZIONE (aggiunta)
# ============================================================
# La presentazione Beamer usa una figura a pannello singolo (solo HICP)
# invece del pannello 2x2 di Cella 7/9. Generata qui per coerenza.

fig, ax = plt.subplots(figsize=(9, 5.5))

plot_asym_wald(ax, lp_hicp_joint, 'Asymmetric Transmission: HICP Inflation',
               'Delta HICP (pp per 25bp shock)')

plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/output/lp_irf_hicp_asym.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura a pannello singolo salvata: lp_irf_hicp_asym.png")

In [ ]:
# ============================================================
# CELL 17 — SALVA RISULTATI FINALI
# ============================================================

lp_hicp_joint.to_csv(f'{DRIVE_ROOT}/output/lp_results_hicp.csv', index=False)
lp_hicp_excl2022.to_csv(f'{DRIVE_ROOT}/output/lp_results_hicp_excl2022.csv', index=False)

print("Risultati LP salvati")
print(f"Righe (baseline): {len(lp_hicp_joint)}")
print(lp_hicp_joint[['h','beta_pos','beta_neg','wald_pval','asymmetric']].to_string(index=False))